In [1]:
import sympy as sp
from SymEigen import *
import diff_friction
import sys
sys.path.append('../')
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction('__host__ __device__')
Gen.DisableLatexComment()

N = 4
Xn = Eigen.Matrix('Xn', 3 * N, 1)
X = Eigen.Matrix('X', 3 * N, 1)
dX = X - Xn
dfdtau = Eigen.Matrix('dfdtau', 2, 1)

Cl = Gen.Closure(Xn)

E0 = Xn[0:3, :]
E1 = Xn[3:6, :]
E2 = Xn[6:9, :]
E3 = Xn[9:12, :]

ea = E1 - E0
eb = E3 - E2
n = ea.cross(eb)
r = E2 - E0

e1 = Eigen.Normalize(ea)
e2 = Eigen.Normalize(n.cross(e1))

D = ea.dot(ea) * eb.dot(eb) - (ea.dot(eb))**2
D1 = (r.dot(ea)) * eb.dot(eb) - (eb.dot(ea)) * (r.dot(eb))
D2 = -ea.dot(ea) * (r.dot(eb)) + (r.dot(ea)) * (eb.dot(ea))

beta_0 = D1/D
beta_1 = D2/D

In [2]:
T = sp.Matrix.zeros(2, 3 * N)
T[0, 0:3] = (1 - beta_0) * e1.T
T[1, 0:3] = (1 - beta_0) * e2.T

T[0, 3:6] = beta_0 * e1.T
T[1, 3:6] = beta_0 * e2.T

T[0, 6:9] = -(1-beta_1) * e1.T
T[1, 6:9] = -(1-beta_1) * e2.T

T[0, 9:12] = - beta_1 * e1.T
T[1, 9:12] = - beta_1 * e2.T

In [3]:
Cl = Gen.Closure(Xn)
content = ''
content+= Cl('TR',T) + '\n'

In [4]:
dTdXn = diff_friction.compute_dTdXn(T, Xn)
dTdXn.shape

(12, 2, 12)

In [5]:
A, B = diff_friction.compute_AB(dTdXn, dX, dfdtau)
A.shape, B.shape

((2, 12), (12, 12))

In [6]:
Cl = Gen.Closure(Xn, X)
content += Cl('An', A) + '\n'

In [ ]:
Cl = Gen.Closure(Xn, X, dfdtau)
content += Cl('Bn', B) + '\n'

In [ ]:
filename = 'ipc_diff_friction_EE'
cuda_backend_source_dir = backend_source_dir('cuda')
f = open(f'{cuda_backend_source_dir}/contact_system/contact_models/sym/{filename}.inl', 'w')
# print(f'{cuda_backend_source_dir}/contact_system/contact_models/sym/{filename}.inl')
f.write(content)
f.close()